In [2]:
import math
import sys
import time
import numpy as np

rng = np.random.default_rng(42)

def measure(fn, x, repeat=3):
    best = float("inf")
    result = None
    for _ in range(repeat):
        t0 = time.perf_counter()
        result = fn(x)
        t1 = time.perf_counter()
        best = min(best, t1 - t0)
    return result, best

In [ ]:
# Задача 1
def loop_sum(a):
    s = 0.0
    for x in a:
        s += x
    return s


def numpy_sum(a):
    return a.sum()


print(f"{'n':<10} | {'Цикл (мс)':<12} | {'NumPy (мс)':<12} | {'Ускорение':<10}")
print("-" * 52)

for n in [10**5, 10**6, 10**7]:
    a = rng.normal(size=n)
    r_loop, t_loop = measure(loop_sum, a)
    r_numpy, t_numpy = measure(numpy_sum, a)
    print(f"{n:<10} | {t_loop*1000:<12.2f} | {t_numpy*1000:<12.2f} | {t_loop / t_numpy:.1f}x")
    assert np.isclose(r_loop, r_numpy)


n          | Цикл (мс)    | NumPy (мс)   | Ускорение 
----------------------------------------------------
100000     | 24.74        | 0.06         | 394.5x
1000000    | 240.60       | 0.63         | 381.2x
10000000   | 1202.02      | 7.28         | 165.1x


In [ ]:
# Задача 2
a = rng.normal(size=10**6)


def loop_mean_std(a):
    s = 0.0
    s_sq = 0.0
    for x in a:
        s += x
        s_sq += x * x
    n = len(a)
    mean = s / n
    std = math.sqrt((s_sq / n) - (mean * mean))
    return mean, std


def numpy_mean_std(a):
    return a.mean(), a.std()


r_loop, t_loop = measure(loop_mean_std, a)
r_numpy, t_numpy = measure(numpy_mean_std, a)

print(f"цикл:      {t_loop*1000:10.2f} мс")
print(f"NumPy:     {t_numpy*1000:10.2f} мс")
print(f"ускорение: {t_loop / t_numpy:.1f}x")
print(f"результаты совпадают (1e-12): {np.allclose(r_loop, r_numpy, atol=1e-12, rtol=1e-12)}")


цикл:          252.86 мс
NumPy:           2.53 мс
ускорение: 100.0x
результаты совпадают (1e-12): True


In [ ]:
# Задача 3
a = rng.normal(size=10**6)


def loop_transform(a):
    res = [0.0] * len(a)
    for i in range(len(a)):
        x = a[i]
        res[i] = math.sin(x) + x * x
    return res


def numpy_transform(a):
    return np.sin(a) + a**2


r_loop, t_loop = measure(loop_transform, a)
r_numpy, t_numpy = measure(numpy_transform, a)

print(f"цикл:      {t_loop*1000:10.2f} мс")
print(f"NumPy:     {t_numpy*1000:10.2f} мс")
print(f"ускорение: {t_loop / t_numpy:.1f}x")
print(f"результаты совпадают: {np.allclose(r_loop, r_numpy)}")


цикл:          361.99 мс
NumPy:          20.62 мс
ускорение: 17.6x
результаты совпадают: True


In [ ]:
# Задача 4
a = rng.normal(size=10**6)

def loop_clip(a):
    res = [0.0] * len(a)
    for i in range(len(a)):
        x = a[i]
        if x < 0:
            res[i] = 0.0
        elif x > 1:
            res[i] = 1.0
        else:
            res[i] = x
    return res

def numpy_clip(a):
    return np.clip(a, 0, 1)

r_loop, t_loop = measure(loop_clip, a)
r_numpy, t_numpy = measure(numpy_clip, a)
changed = np.sum((a < 0) | (a > 1))

print(f"цикл:      {t_loop*1000:10.2f} мс")
print(f"NumPy:     {t_numpy*1000:10.2f} мс")
print(f"ускорение: {t_loop / t_numpy:.1f}x")
print(f"результаты совпадают: {np.allclose(r_loop, r_numpy)}")
print(f"изменено элементов: {changed}")

In [3]:
# Задача 5
m = rng.normal(size=(10**5, 10))


def loop_row_sum(m):
    n = m.shape[0]
    result = [0.0] * n
    for i in range(n):
        s = 0.0
        for j in range(10):
            s += m[i, j]
        result[i] = s
    return result


def numpy_row_sum(m):
    return m.sum(axis=1)


r_loop, t_loop = measure(loop_row_sum, m)
r_numpy, t_numpy = measure(numpy_row_sum, m)

print(f"цикл:      {t_loop*1000:10.2f} мс")
print(f"NumPy:     {t_numpy*1000:10.2f} мс")
print(f"ускорение: {t_loop / t_numpy:.1f}x")
print(f"результаты совпадают: {np.allclose(r_loop, r_numpy)}")


цикл:          454.91 мс
NumPy:           3.75 мс
ускорение: 121.2x
результаты совпадают: True


In [4]:
# Задача 6
m = rng.normal(size=(10**5, 10))


def loop_col_norm(m):
    rows, cols = m.shape
    means = [0.0] * cols
    for j in range(cols):
        s = 0.0
        for i in range(rows):
            s += m[i, j]
        means[j] = s / rows

    res = [[0.0] * cols for _ in range(rows)]
    for i in range(rows):
        for j in range(cols):
            res[i][j] = m[i, j] - means[j]
    return res


def numpy_col_norm(m):
    return m - m.mean(axis=0)


r_loop, t_loop = measure(loop_col_norm, m)
r_numpy, t_numpy = measure(numpy_col_norm, m)

print(f"цикл:      {t_loop*1000:10.2f} мс")
print(f"NumPy:     {t_numpy*1000:10.2f} мс")
print(f"ускорение: {t_loop / t_numpy:.1f}x")
print(f"результаты совпадают: {np.allclose(r_loop, r_numpy)}")


цикл:          591.51 мс
NumPy:           5.89 мс
ускорение: 100.4x
результаты совпадают: True


In [5]:
# Задача 7
a = rng.normal(size=10**6)


def loop_argmax(a):
    max_val = a[0]
    max_idx = 0
    for i in range(1, len(a)):
        if a[i] > max_val:
            max_val = a[i]
            max_idx = i
    return max_val, max_idx


def numpy_argmax(a):
    return a.max(), a.argmax()


r_loop, t_loop = measure(loop_argmax, a)
r_numpy, t_numpy = measure(numpy_argmax, a)
eq = np.isclose(r_loop[0], r_numpy[0]) and (r_loop[1] == r_numpy[1])

print(f"цикл:      {t_loop*1000:10.2f} мс")
print(f"NumPy:     {t_numpy*1000:10.2f} мс")
print(f"ускорение: {t_loop / t_numpy:.1f}x")
print(f"результаты совпадают: {eq}")


цикл:          131.85 мс
NumPy:           0.97 мс
ускорение: 136.2x
результаты совпадают: True


In [6]:
# Задача 8
a = rng.normal(size=10**6)


def loop_filter(a):
    return [x for x in a if x > 0]


def numpy_filter(a):
    return a[a > 0]


r_loop, t_loop = measure(loop_filter, a)
r_numpy, t_numpy = measure(numpy_filter, a)

mem_list_mb = (sys.getsizeof(r_loop) + len(r_loop) * 28) / (1024**2)
mem_np_mb = r_numpy.nbytes / (1024**2)

print(f"цикл:      {t_loop*1000:10.2f} мс")
print(f"NumPy:     {t_numpy*1000:10.2f} мс")
print(f"ускорение: {t_loop / t_numpy:.1f}x")
print(f"результаты совпадают: {np.allclose(r_loop, r_numpy)}")
print(f"память: список ~{mem_list_mb:.2f} МБ vs NumPy {mem_np_mb:.2f} МБ")


цикл:          208.23 мс
NumPy:          10.29 мс
ускорение: 20.2x
результаты совпадают: True
память: список ~17.35 МБ vs NumPy 3.82 МБ
